# RSNA Knee Metadata Inspection

Use this notebook locally or on Kaggle to understand the CSV metadata before training a model.

In [ ]:
from pathlib import Path
import pandas as pd

# Locate the repo root whether the notebook runs from notebooks/ or from the root.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'data').exists():
    REPO_ROOT = REPO_ROOT.parent

# Use the full Kaggle competition data when running on Kaggle,
# otherwise the CSVs versioned in this repo.
KAGGLE_ROOT = Path('/kaggle/input/rsna-knee-abnormality-detection')
DATA_ROOT = KAGGLE_ROOT if KAGGLE_ROOT.exists() else REPO_ROOT / 'data' / 'raw'

# With the mini zip, metadata is under DATA_ROOT / 'metadata'.
# With the full Kaggle data, it is directly under DATA_ROOT.
META_ROOT = DATA_ROOT / 'metadata' if (DATA_ROOT / 'metadata').exists() else DATA_ROOT

print('DATA_ROOT:', DATA_ROOT.resolve())
print('META_ROOT:', META_ROOT.resolve())
print('META_ROOT exists:', META_ROOT.exists())

In [ ]:
def read_first_existing(*names):
    for name in names:
        path = META_ROOT / name
        if path.exists():
            print('Reading:', path)
            return pd.read_csv(path)
    print('Missing all of:', names)
    return pd.DataFrame()

# This is one row per knee MRI study (multiple series).
train_df = read_first_existing('train.csv', 'train_subset.csv')
# This is one row per MRI series
series_df = read_first_existing('train_series.csv', 'train_series_subset.csv')
test_df = read_first_existing('test.csv')
test_series_df = read_first_existing('test_series.csv')

print('train_df:', train_df.shape)
print('series_df:', series_df.shape)
print('test_df:', test_df.shape)
print('test_series_df:', test_series_df.shape)

In [ ]:
display(train_df.head())
display(series_df.head())
display(test_df.head())
display(test_series_df.head())

There are almost no explicit labels in train.csv, only surgeon's reports

train_series.csv tells us what image series exist for each study:
StudyInstanceUID
SeriesInstanceUID
Fluid_Sensitive
Fat_Suppression
Anatomical_Plane

In [ ]:
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
present_targets = [c for c in TARGETS if c in train_df.columns]

print('Target columns found:', present_targets)
if present_targets:
    summary = train_df[present_targets].agg(['count', 'mean', 'sum']).T
    summary = summary.rename(columns={'mean': 'positive_rate', 'sum': 'positive_count'})
    display(summary.sort_values('positive_rate', ascending=False))
else:
    print('No explicit target columns found. Labels may need to be extracted from reports or another file.')

Only 58 studies have actual 0 or 1 per target, idk if its useful for other case because the labels will be learn from reports then, but maybe to understand the reports these 58 labels could be helpful 

In [ ]:
if not series_df.empty:
    print('Series columns:', list(series_df.columns))
    for col in ['Anatomical_Plane', 'Fat_Suppression', 'Fluid_Sensitive']:
        if col in series_df.columns:
            print('\n', col)
            display(series_df[col].value_counts(dropna=False))

    if 'StudyInstanceUID' in series_df.columns:
        series_per_study = series_df.groupby('StudyInstanceUID').size()
        display(series_per_study.describe())
        display(series_per_study.value_counts().sort_index().head(20).rename('n_studies'))

So, for each study we should find available sagittal/coronal/axial studies and shoose fluid/non-fluid series since number of series is not unified across studies

so obvious ofc...............

In [ ]:
report_cols = [c for c in train_df.columns if 'report' in c.lower()]
print('Report-like columns:', report_cols)

for col in report_cols:
    non_null = train_df[col].dropna().astype(str)
    print(f'{col}: non-null={len(non_null)}, avg_chars={non_null.str.len().mean():.1f}' if len(non_null) else f'{col}: empty')
    display(non_null.head(5).to_frame())

the reports display some broken characters, there are multilingual texts --> test reading with UTF-8 and possibly repair text with something like ``ftfy``